# 🚀 BCTC & BCTN UNIFIED MULTI-THREADED PIPELINE (1 TÀI KHOẢN DUY NHẤT)
### Bóc tách dữ liệu BCTC & Báo cáo thường niên cho toàn bộ 1.300+ mã cổ phiếu

Notebook này được tối ưu để **chỉ cần chạy trên 1 tài khoản Google Colab duy nhất**:
- ⚡ **Đa luồng cực nhanh (8 Threads)**: Tận dụng đường truyền 10Gbps của Google Colab để tải và bóc tách song song 8 mã cùng lúc.
- 🔄 **Kế thừa 100% tiến độ cũ**: Tự động quét và nạp toàn bộ các file `bctc_shard_*.json` (từ 5 tài khoản bạn vừa chạy) + `extracted_bctc_lake.json` cũ. Những mã nào đã cào xong sẽ **bỏ qua ngay (Skip)**, chỉ chạy tiếp các mã còn thiếu!
- 💾 **Lưu trực tiếp vào Data Lake**: Tự động cập nhật file tổng `extracted_bctc_lake.json` sau mỗi 10 mã.

> **CÁCH DÙNG**: Chỉ cần bấm menu **Runtime** -> **Run all** (hoặc nhấn `Ctrl + F9`). Không cần chia tài khoản hay chỉnh sửa bất kỳ biến nào nữa!

In [ ]:
# =============================================================================
# 1. CÀI ĐẶT THƯ VIỆN BÓC TÁCH DỮ LIỆU ĐA LUỒNG
# =============================================================================
!pip install -q pdfplumber beautifulsoup4 requests tqdm vnstock
print("✅ Môi trường bóc tách đa luồng đã sẵn sàng!")

In [ ]:
# =============================================================================
# 2. KẾT NỐI GOOGLE DRIVE & THIẾT LẬP KHO DỮ LIỆU
# =============================================================================
import os, sys, json, time, re, shutil, threading
from typing import Dict, List, Any, Optional
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import drive
from tqdm import tqdm
import requests
from bs4 import BeautifulSoup
import pdfplumber

# Mount Google Drive
drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/vnstock_data'
PDF_LAKE_DIR = os.path.join(BASE_DIR, 'pdf_lake')
os.makedirs(PDF_LAKE_DIR, exist_ok=True)

SCRATCH_DIR = '/content/temp_pdf_scratch'
os.makedirs(SCRATCH_DIR, exist_ok=True)

# File Data Lake chính
LAKE_FILE = os.path.join(PDF_LAKE_DIR, "extracted_bctc_lake.json")

# CẤU HÌNH ĐỘ SÂU & ĐA LUỒNG
MAX_WORKERS = 8           # Chạy song song 8 luồng tải & bóc tách
MAX_BCTC_PER_SYMBOL = 8   # 8 quý gần nhất (~2 năm, đủ TTM & tăng trưởng)
MAX_BCTN_PER_SYMBOL = 2   # 2 Báo cáo thường niên gần nhất

print(f"📁 Kho dữ liệu Google Drive: {PDF_LAKE_DIR}")
print(f"⚡ Cấu hình: {MAX_WORKERS} Threads song song | Tối đa {MAX_BCTC_PER_SYMBOL} BCTC + {MAX_BCTN_PER_SYMBOL} BCTN mỗi mã")

In [ ]:
# =============================================================================
# 3. TỰ ĐỘNG GỘP CÁC SHARD CŨ VÀ TẢI DANH MỤC TOÀN THỊ TRƯỜNG
# =============================================================================
# 1. Nạp và gộp tất cả dữ liệu từ các file shard cũ (từ 5 máy bạn vừa chạy)
lake_data = {}
if os.path.exists(LAKE_FILE):
    try:
        with open(LAKE_FILE, 'r', encoding='utf-8') as f:
            lake_data = json.load(f)
        print(f"📚 Đã nạp {len(lake_data)} mã từ file tổng extracted_bctc_lake.json")
    except Exception:
        lake_data = {}

# Quét các shard cũ bctc_shard_*.json để kế thừa không sót mã nào
shards_found = [f for f in os.listdir(PDF_LAKE_DIR) if re.match(r'^bctc_shard_\d+\.json$', f)]
for s_fname in shards_found:
    s_path = os.path.join(PDF_LAKE_DIR, s_fname)
    try:
        with open(s_path, 'r', encoding='utf-8') as f:
            s_content = json.load(f)
        for sym, data in s_content.items():
            if sym not in lake_data or len(data.get('periods', [])) >= len(lake_data.get(sym, {}).get('periods', [])):
                lake_data[sym] = data
        print(f"  🔄 Kế thừa thành công từ {s_fname}")
    except Exception as e:
        pass

print(f"💎 Tổng số mã đã có dữ liệu sẵn: {len(lake_data)} mã")

# 2. Lấy danh mục 1.300+ mã toàn thị trường
all_symbols = []
symbols_cache_file = os.path.join(BASE_DIR, "all_symbols.json")
if os.path.exists(symbols_cache_file):
    try:
        with open(symbols_cache_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
            if isinstance(data, list):
                for item in data:
                    if isinstance(item, dict):
                        s = item.get('symbol') or item.get('ticker')
                        if s: all_symbols.append(s)
                    elif isinstance(item, str):
                        all_symbols.append(item)
    except Exception:
        pass

if not all_symbols:
    try:
        from vnstock import Listing
        lst = Listing()
        df = lst.all_symbols()
        if df is not None and not df.empty:
            col = 'symbol' if 'symbol' in df.columns else 'ticker'
            all_symbols = df[col].dropna().unique().tolist()
    except Exception:
        pass

all_symbols = sorted(list(set([str(s).strip().upper() for s in all_symbols if s and len(str(s).strip()) == 3 and str(s).strip().isalnum()])))
print(f"📊 Toàn thị trường: {len(all_symbols)} mã")

# 3. Lọc danh sách thực sự còn thiếu cần chạy tiếp
remaining_symbols = [
    s for s in all_symbols
    if s not in lake_data or len(lake_data[s].get('periods', [])) < MAX_BCTC_PER_SYMBOL
]
print(f"⚡ SỐ MÃ CÒN LẠI CẦN CHẠY: {len(remaining_symbols)}/{len(all_symbols)} mã (Đã có sẵn {len(all_symbols) - len(remaining_symbols)} mã!)")

In [ ]:
# =============================================================================
# 4. PARSER ENGINE CHUẨN THÔNG TƯ 200 & BÁO CÁO THƯỜNG NIÊN
# =============================================================================
def clean_num(val_str):
    if not val_str: return 0.0
    s = str(val_str).strip()
    is_neg = s.startswith('(') and s.endswith(')')
    s = re.sub(r'[^0-9,\.-]', '', s)
    if not s: return 0.0
    if '.' in s and ',' in s:
        if s.find('.') < s.find(','):
            s = s.replace('.', '').replace(',', '.')
        else:
            s = s.replace(',', '')
    elif ',' in s and '.' not in s:
        parts = s.split(',')
        if len(parts[-1]) == 3:
            s = s.replace(',', '')
        else:
            s = s.replace(',', '.')
    try:
        v = float(s)
        return -v if is_neg else v
    except:
        return 0.0

def detect_period(text):
    t = text.lower()
    quarter = None
    q_m = re.search(r'qu[ýy]\s*([1-4iv]+)', t)
    if q_m:
        q_raw = q_m.group(1).upper()
        mapping = {'1': 1, 'I': 1, '2': 2, 'II': 2, '3': 3, 'III': 3, '4': 4, 'IV': 4}
        quarter = mapping.get(q_raw)
    elif 'bán niên' in t or '6 tháng' in t:
        quarter = '6M'
    elif 'cả năm' in t or 'năm tài chính' in t:
        quarter = 'FY'
    year = None
    y_m = re.search(r'(?:năm|ktkt)\s*(20[12][0-9])', t)
    if y_m: year = int(y_m.group(1))
    is_audited = any(k in t for k in ['kiểm toán', 'đã kiểm toán', 'soát xét'])
    return {'quarter': quarter, 'year': year, 'is_audited': is_audited}

def extract_bctc_pdf(pdf_path):
    records = {}
    try:
        with pdfplumber.open(pdf_path) as pdf:
            full_text = " ".join([page.extract_text() or "" for page in pdf.pages[:8]])
            period = detect_period(full_text)
            for page in pdf.pages:
                tables = page.extract_tables()
                for tbl in tables:
                    if not tbl: continue
                    for row in tbl:
                        if not row or len(row) < 3: continue
                        text_line = " ".join([str(c) for c in row if c]).lower()
                        if 'doanh thu thuần' in text_line or 'mã số 10' in text_line:
                            val = clean_num(row[-1] or (row[-2] if len(row) > 2 else 0))
                            if val > 0 and 'revenue' not in records: records['revenue'] = val
                        elif 'lợi nhuận sau thuế' in text_line or 'mã số 60' in text_line:
                            val = clean_num(row[-1] or (row[-2] if len(row) > 2 else 0))
                            if val != 0 and 'net_profit' not in records: records['net_profit'] = val
                        elif 'lưu chuyển tiền thuần từ hoạt động kinh doanh' in text_line or 'mã số 20' in text_line:
                            val = clean_num(row[-1] or (row[-2] if len(row) > 2 else 0))
                            if val != 0 and 'cfo' not in records: records['cfo'] = val
                        elif 'mua sắm' in text_line and ('tscđ' in text_line or 'tài sản cố định' in text_line):
                            val = abs(clean_num(row[-1] or (row[-2] if len(row) > 2 else 0)))
                            if val > 0 and 'capex' not in records: records['capex'] = val
                        elif 'tổng cộng tài sản' in text_line or 'mã số 270' in text_line:
                            val = clean_num(row[-1] or (row[-2] if len(row) > 2 else 0))
                            if val > 0 and 'total_assets' not in records: records['total_assets'] = val
                        elif 'vốn chủ sở hữu' in text_line or 'mã số 400' in text_line:
                            val = clean_num(row[-1] or (row[-2] if len(row) > 2 else 0))
                            if val > 0 and 'equity' not in records: records['equity'] = val
            if 'cfo' in records and 'capex' in records:
                records['fcf'] = records['cfo'] - records['capex']
            records['period'] = period
            return records
    except Exception:
        return {}

def extract_bctn_targets(pdf_path):
    targets = {}
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for i, page in enumerate(pdf.pages[:100]):
                t = (page.extract_text() or "").lower()
                if 'kế hoạch kinh doanh' in t or 'kế hoạch năm' in t:
                    rev_m = re.search(r'doanh\s*thu.*?([0-9]{1,3}(?:[\.,][0-9]{3})+|\d+)\s*(?:tỷ|triệu)?', t)
                    if rev_m and 'plan_revenue' not in targets: targets['plan_revenue'] = rev_m.group(1)
                    prof_m = re.search(r'lợi\s*nhuận.*?([0-9]{1,3}(?:[\.,][0-9]{3})+|\d+)\s*(?:tỷ|triệu)?', t)
                    if prof_m and 'plan_profit' not in targets: targets['plan_profit'] = prof_m.group(1)
                if 'cổ tức' in t:
                    div_m = re.search(r'cổ\s*tức.*?(\d{1,2}(?:[\.,]\d+)?\s*%)', t)
                    if div_m and 'expected_dividend' not in targets: targets['expected_dividend'] = div_m.group(1)
                if len(targets) >= 3: break
    except Exception:
        pass
    return targets

In [ ]:
# =============================================================================
# 5. CRAWLER LẤY TÀI LIỆU & HÀM XỬ LÝ 1 MÃ (ĐỘC LẬP THEO LUỒNG)
# =============================================================================
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

def get_symbol_pdf_links(symbol):
    url = f"https://s.cafef.vn/Ajax/Events_RelatedNews_New.aspx?symbol={symbol}&floorID=0&configID=0&PageIndex=1&PageSize=30&Type=2"
    docs = []
    try:
        res = requests.get(url, headers=HEADERS, timeout=10)
        if res.status_code == 200:
            soup = BeautifulSoup(res.text, 'html.parser')
            for a in soup.find_all('a', href=True):
                href = a['href']
                title = a.get_text().strip()
                if href.lower().endswith('.pdf'):
                    full_url = href if href.startswith('http') else 'https://s.cafef.vn' + href
                    doc_type = 'BCTN' if ('thường niên' in title.lower() or 'annual' in title.lower()) else 'BCTC'
                    docs.append({'title': title, 'url': full_url, 'type': doc_type})
    except Exception:
        pass
    return docs

def process_one_symbol(sym, thread_id):
    docs = get_symbol_pdf_links(sym)
    sym_result = lake_data.get(sym, {'symbol': sym, 'periods': [], 'annual_report_targets': {}, 'updated_at': int(time.time())})
    bctc_docs = [d for d in docs if d['type'] == 'BCTC'][:MAX_BCTC_PER_SYMBOL]
    bctn_docs = [d for d in docs if d['type'] == 'BCTN'][:MAX_BCTN_PER_SYMBOL]
    existing_titles = set([p.get('doc_title', '') for p in sym_result.get('periods', [])])
    
    # Bóc BCTC
    for d in bctc_docs:
        if d['title'] in existing_titles: continue
        temp_pdf = os.path.join(SCRATCH_DIR, f"{sym}_{thread_id}_bctc.pdf")
        try:
            r = requests.get(d['url'], headers=HEADERS, timeout=20)
            if r.status_code == 200:
                with open(temp_pdf, 'wb') as f: f.write(r.content)
                parsed = extract_bctc_pdf(temp_pdf)
                if parsed:
                    parsed['doc_title'] = d['title']
                    sym_result['periods'].append(parsed)
        except Exception:
            pass
        finally:
            if os.path.exists(temp_pdf): os.remove(temp_pdf)
            
    # Bóc Báo cáo thường niên
    if not sym_result.get('annual_report_targets'):
        for d in bctn_docs:
            temp_pdf = os.path.join(SCRATCH_DIR, f"{sym}_{thread_id}_bctn.pdf")
            try:
                r = requests.get(d['url'], headers=HEADERS, timeout=30)
                if r.status_code == 200:
                    with open(temp_pdf, 'wb') as f: f.write(r.content)
                    targets = extract_bctn_targets(temp_pdf)
                    if targets:
                        sym_result['annual_report_targets'] = targets
                        break
            except Exception:
                pass
            finally:
                if os.path.exists(temp_pdf): os.remove(temp_pdf)
                
    return sym, sym_result

In [ ]:
# =============================================================================
# 6. ĐIỀU PHỐI ĐA LUỒNG & AUTO-SAVE CHECKPOINT VÀO DATA LAKE TỔNG
# =============================================================================
save_lock = threading.Lock()
completed_counter = 0

def save_checkpoint():
    tmp_save = LAKE_FILE + ".tmp"
    with open(tmp_save, 'w', encoding='utf-8') as f:
        json.dump(lake_data, f, ensure_ascii=False, indent=2)
    os.replace(tmp_save, LAKE_FILE)

print(f"🚀 BẮT ĐẦU CHẠY ĐA LUỒNG ({MAX_WORKERS} WORKERS) CHO {len(remaining_symbols)} MÃ CÒN LẠI...")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    future_map = {
        executor.submit(process_one_symbol, sym, idx % MAX_WORKERS): sym
        for idx, sym in enumerate(remaining_symbols)
    }
    
    pbar = tqdm(as_completed(future_map), total=len(remaining_symbols), desc="Unified Extraction")
    for future in pbar:
        try:
            sym, result = future.result()
            with save_lock:
                lake_data[sym] = result
                completed_counter += 1
                if completed_counter % 10 == 0 or completed_counter == len(remaining_symbols):
                    save_checkpoint()
        except Exception as err:
            pass

# Lưu hoàn chỉnh lần cuối
save_checkpoint()
print("=" * 70)
print(f"🎉 HOÀN THÀNH TOÀN BỘ! ĐÃ BÓC TÁCH VÀ LƯU {len(lake_data)} MÃ VÀO GOOGLE DRIVE!")
print(f"📁 File dữ liệu tổng: {LAKE_FILE} ({round(os.path.getsize(LAKE_FILE)/(1024*1024), 2)} MB)")
print("=" * 70)